IMPORT IMPORTANT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

LOAD THE DATASET

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DatasetAuspify.csv")

In [ ]:
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [ ]:
df.shape

(8790, 10)

EXPLORATORY DATA ANALYSIS(EDA)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
release_year,8790.0,2014.183163,8.825466,1925.0,2013.0,2017.0,2019.0,2021.0


In [ ]:
df["type"].value_counts()

,count
type,
Movie,6126
TV Show,2664


In [ ]:
df["listed_in"].value_counts().head(20)

,count
listed_in,
"Dramas, International Movies",362
Documentaries,359
Stand-Up Comedy,334
"Comedies, Dramas, International Movies",274
"Dramas, Independent Movies, International Movies",252
Kids' TV,219
Children & Family Movies,215
"Children & Family Movies, Comedies",201
"Documentaries, International Movies",186


In [ ]:
df["listed_in"].head(10)

,listed_in
0,Documentaries
1,"Crime TV Shows, International TV Shows, TV Act..."
2,"TV Dramas, TV Horror, TV Mysteries"
3,"Children & Family Movies, Comedies"
4,"Dramas, Independent Movies, International Movies"
5,"British TV Shows, Reality TV"
6,"Comedies, Dramas"
7,"Children & Family Movies, Comedies, Music & Mu..."
8,"Dramas, International Movies"
9,"Children & Family Movies, Music & Musicals"


In [ ]:
genres = df["listed_in"].str.split(", ")
genres.head()

,listed_in
0,[Documentaries]
1,"[Crime TV Shows, International TV Shows, TV Ac..."
2,"[TV Dramas, TV Horror, TV Mysteries]"
3,"[Children & Family Movies, Comedies]"
4,"[Dramas, Independent Movies, International Mov..."


In [ ]:
genre_counts = genres.explode().value_counts()
genre_counts.head(10)

,count
listed_in,
International Movies,2752
Dramas,2426
Comedies,1674
International TV Shows,1349
Documentaries,869
Action & Adventure,859
TV Dramas,762
Independent Movies,756
Children & Family Movies,641


FEATURE ENGINEERING

In [ ]:
df["content"] = df["type"] + " " + df["listed_in"]

df[["title", "type", "listed_in", "content"]].head()

,title,type,listed_in,content
0,Dick Johnson Is Dead,Movie,Documentaries,Movie Documentaries
1,Ganglands,TV Show,"Crime TV Shows, International TV Shows, TV Act...","TV Show Crime TV Shows, International TV Shows..."
2,Midnight Mass,TV Show,"TV Dramas, TV Horror, TV Mysteries","TV Show TV Dramas, TV Horror, TV Mysteries"
3,Confessions of an Invisible Girl,Movie,"Children & Family Movies, Comedies","Movie Children & Family Movies, Comedies"
4,Sankofa,Movie,"Dramas, Independent Movies, International Movies","Movie Dramas, Independent Movies, Internationa..."


SCIKITY LEARN VS NLP

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["content"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (8790, 45)


In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (8790, 8790)


In [ ]:

indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

print(indices.head())

title
Dick Johnson Is Dead                0
Ganglands                           1
Midnight Mass                       2
Confessions of an Invisible Girl    3
Sankofa                             4
dtype: int64


RECOMMENDATION FUNCTION

In [ ]:

def recommend(title, top_n=5):
    if title not in indices:
        return f"Title '{title}' Not Found."

    idx = indices[title]

    scores = list(enumerate(similarity_matrix[idx]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    scores = [
        item for item in scores
        if item[0] != idx
    ]

    top_scores = scores[:top_n]

    recommendations = []

    for i, score in top_scores:
        recommendations.append({
            "title": df.iloc[i]["title"],
            "type": df.iloc[i]["type"],
            "genres": df.iloc[i]["listed_in"],
            "similarity_score": round(float(score), 3)
        })

    return pd.DataFrame(recommendations)

In [ ]:

recommend("Kota Factory")

,title,type,genres,similarity_score
0,Hotel Del Luna,TV Show,"International TV Shows, Romantic TV Shows, TV ...",1.0
1,Hometown Cha-Cha-Cha,TV Show,"International TV Shows, Romantic TV Shows, TV ...",1.0
2,The Smart Money Woman,TV Show,"International TV Shows, Romantic TV Shows, TV ...",1.0
3,Feels Like Ishq,TV Show,"International TV Shows, Romantic TV Shows, TV ...",1.0
4,Generation 56k,TV Show,"International TV Shows, Romantic TV Shows, TV ...",1.0


CHECK FOR THE RECOMMENDED GENRES

In [ ]:

df.loc[
    df["title"].isin([
        "Kota Factory",
        "Hotel Del Luna",
        "Hometown Cha-Cha-Cha",
        "The Smart Money Woman",
        "Feels Like Ishq",
        "Generation 56k"
    ]),
    ["title", "type", "listed_in", "content"]
]

,title,type,listed_in,content
43,Hotel Del Luna,TV Show,"International TV Shows, Romantic TV Shows, TV ...","TV Show International TV Shows, Romantic TV Sh..."
49,Hometown Cha-Cha-Cha,TV Show,"International TV Shows, Romantic TV Shows, TV ...","TV Show International TV Shows, Romantic TV Sh..."
58,The Smart Money Woman,TV Show,"International TV Shows, Romantic TV Shows, TV ...","TV Show International TV Shows, Romantic TV Sh..."
104,Feels Like Ishq,TV Show,"International TV Shows, Romantic TV Shows, TV ...","TV Show International TV Shows, Romantic TV Sh..."
123,Generation 56k,TV Show,"International TV Shows, Romantic TV Shows, TV ...","TV Show International TV Shows, Romantic TV Sh..."
6572,Kota Factory,TV Show,"International TV Shows, Romantic TV Shows, TV ...","TV Show International TV Shows, Romantic TV Sh..."


LOOKING RECOMMENDETIONS FOR OTHER TITLES

In [ ]:

recommend("Sankofa")

,title,type,genres,similarity_score
0,Ankahi Kahaniya,Movie,"Dramas, Independent Movies, International Movies",1.0
1,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies",1.0
2,La diosa del asfalto,Movie,"Dramas, Independent Movies, International Movies",1.0
3,Eyes of a Thief,Movie,"Dramas, Independent Movies, International Movies",1.0
4,Pomegranates and Myrrh,Movie,"Dramas, Independent Movies, International Movies",1.0


LAST CHECKUP

In [ ]:

result = recommend("Sankofa", top_n=5)

print("Number of recommendations:", len(result))
print("Sankofa included:", "Sankofa" in result["title"].values)

result

Number of recommendations: 5
Sankofa included: False


,title,type,genres,similarity_score
0,Ankahi Kahaniya,Movie,"Dramas, Independent Movies, International Movies",1.0
1,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies",1.0
2,La diosa del asfalto,Movie,"Dramas, Independent Movies, International Movies",1.0
3,Eyes of a Thief,Movie,"Dramas, Independent Movies, International Movies",1.0
4,Pomegranates and Myrrh,Movie,"Dramas, Independent Movies, International Movies",1.0
